# Batch Evaluate New Datasets (Forecasting)

Notebook version of `scripts/eval_new_datasets_forecasting.py`.

What it does:
1. Load one checkpoint
2. Evaluate multiple datasets
3. Compute test NLL
4. Optionally run sampling-based forecast-count metrics
5. Save JSON + CSV summary


In [5]:
import csv
import json
import sys
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import torch


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd.parent):
        if (candidate / 'src').exists():
            return candidate
    raise FileNotFoundError("Cannot find project root containing 'src' directory.")


PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import src
import src.catalogs as catalogs  # noqa: F401, ensures registrations
from src.train.config_setup import load_and_prepare_model
from src.utils.tpp_experiments import load_tpp_catalog, sample_tpp_forecasts
from src.utils.utils import set_seed

print(f'PROJECT_ROOT: {PROJECT_ROOT}')

PROJECT_ROOT: /root/autodl-tmp/em_eqf


## Parameters
Edit this cell before running.

In [6]:
# --- Required ---
CHECKPOINT_PATH = PROJECT_ROOT / 'checkpoints' / 'etas_20260312-154442' / 'best_model_1.pth'
DATASETS = ['SSFS', 'Basel', 'PNR_1z']

# --- Forecast settings ---
FORECAST_DURATION = 10.0
T_FORECAST = None  # None => use test_seq.t_nll_start
NUM_SAMPLES = 200
SAMPLES_PER_BATCH = 200
SAMPLE_MAX_LENGTH = None  # e.g. 50000
SKIP_FORECAST = True      # True => only compute test_nll (fast)

# --- Runtime ---
SEED = 0
COMPILE_MODEL = False
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# --- Catalog cfg overrides ---
GLOBAL_CATALOG_CFG = {}  # e.g. {'freq': '1D'}
DATASET_CFG_FILE = PROJECT_ROOT / 'config' / 'eval_dataset_cfg.example.json'  # or None

# --- Output ---
OUTPUT_JSON = None  # None => <checkpoint-dir>/batch_eval_metrics.json
OUTPUT_CSV = None   # None => <checkpoint-dir>/batch_eval_metrics.csv

print('DEVICE:', DEVICE)
print('CHECKPOINT_PATH:', CHECKPOINT_PATH)
print('DATASETS:', DATASETS)

DEVICE: cuda:0
CHECKPOINT_PATH: /root/autodl-tmp/em_eqf/checkpoints/etas_20260312-154442/best_model_1.pth
DATASETS: ['SSFS', 'Basel', 'PNR_1z']


In [7]:
def _load_dataset_cfg_map(path: Path | None) -> dict[str, dict[str, Any]]:
    if path is None:
        return {}
    data = json.loads(path.read_text(encoding='utf-8'))
    if not isinstance(data, dict):
        raise ValueError('dataset cfg file must contain a JSON object.')
    out: dict[str, dict[str, Any]] = {}
    for k, v in data.items():
        if not isinstance(v, dict):
            raise ValueError(f"dataset cfg for '{k}' must be a JSON object.")
        out[k] = v
    return out


def _loss_to_scalar(loss_obj: Any) -> float:
    if torch.is_tensor(loss_obj):
        return float(loss_obj.mean().detach().cpu().item())

    if isinstance(loss_obj, dict):
        if 'time' in loss_obj and torch.is_tensor(loss_obj['time']):
            return float(loss_obj['time'].mean().detach().cpu().item())
        tensor_values = [v for v in loss_obj.values() if torch.is_tensor(v)]
        if tensor_values:
            return float(torch.stack([v.mean() for v in tensor_values]).mean().detach().cpu().item())

    raise TypeError(f'Unsupported nll_loss return type: {type(loss_obj)}')


def _build_catalog(dataset_name: str, catalog_cfg: dict[str, Any]):
    return load_tpp_catalog(
        dataset_name,
        base_dir=PROJECT_ROOT / 'data' / dataset_name,
        catalog_cfg=catalog_cfg,
        candidates=[f'{dataset_name}-Standard', dataset_name],
    )


def _generate_forecasts(
    model: torch.nn.Module,
    past_seq: Any,
    duration: float,
    num_samples: int,
    samples_per_batch: int,
    seed: int,
    sample_max_length: int | None,
) -> list[Any]:
    return sample_tpp_forecasts(
        model,
        past_seq,
        duration=duration,
        num_samples=num_samples,
        samples_per_batch=samples_per_batch,
        seed=seed,
        sample_max_length=sample_max_length,
        verbose=False,
    )


def evaluate_one_dataset(
    *,
    model: torch.nn.Module,
    device: torch.device,
    dataset_name: str,
    catalog_cfg: dict[str, Any],
    t_forecast: float | None,
    forecast_duration: float,
    num_samples: int,
    samples_per_batch: int,
    sample_max_length: int | None,
    seed: int,
    skip_forecast: bool,
) -> dict[str, Any]:
    result: dict[str, Any] = {'dataset': dataset_name}

    cat, registry_name, used_init_kwargs = _build_catalog(dataset_name, catalog_cfg)
    result['catalog_registry_name'] = registry_name
    result['catalog_init_kwargs'] = used_init_kwargs

    test_seq = cat.test[0]
    test_batch = src.data.Batch.from_list([test_seq]).to(device)
    with torch.no_grad():
        test_nll_raw = model.nll_loss(test_batch)
        result['test_nll'] = _loss_to_scalar(test_nll_raw)

    result['num_events_test'] = int(len(test_seq))
    result['test_t_start'] = float(test_seq.t_start)
    result['test_t_end'] = float(test_seq.t_end)
    result['test_t_nll_start'] = float(test_seq.t_nll_start)

    if skip_forecast:
        return result

    tf = float(test_seq.t_nll_start) if t_forecast is None else float(t_forecast)
    tf = max(float(test_seq.t_start), min(tf, float(test_seq.t_end)))
    max_duration = float(test_seq.t_end) - tf
    if max_duration <= 0:
        raise ValueError(
            f'No room for forecast window on dataset={dataset_name}. '
            f't_forecast={tf}, test_t_end={test_seq.t_end}.'
        )
    dur = min(float(forecast_duration), max_duration)

    past_seq = test_seq.get_subsequence(test_seq.t_start, tf, reset_t_nll_to_end=True)
    obs_seq = test_seq.get_subsequence(tf, tf + dur, reset_t_nll_to_end=True)

    if device.type == 'cuda':
        past_seq = past_seq.to(device)

    bg_model = getattr(model, 'bg_model', None)
    if (
        bg_model is not None
        and hasattr(bg_model, 'cache_batch')
        and hasattr(test_seq, 'time_series')
        and hasattr(test_seq, 'time_series_times')
    ):
        ts = test_seq.time_series.unsqueeze(0).to(device)
        ts_t = test_seq.time_series_times.unsqueeze(0).to(device)
        bg_model.cache_batch(time_series=ts, time_series_times=ts_t)

    forecasts = _generate_forecasts(
        model=model,
        past_seq=past_seq,
        duration=dur,
        num_samples=num_samples,
        samples_per_batch=samples_per_batch,
        sample_max_length=sample_max_length,
        seed=seed,
    )

    counts = np.array([len(fc) for fc in forecasts], dtype=float)
    obs_count = float(len(obs_seq))
    if counts.size == 0:
        raise RuntimeError('No forecast samples generated.')

    q025, q975 = np.percentile(counts, [2.5, 97.5])
    rmse = float(np.sqrt(np.mean((counts - obs_count) ** 2)))
    coverage = float(q025 <= obs_count <= q975)

    result.update(
        {
            't_forecast': tf,
            'forecast_duration': dur,
            'num_forecasts': int(counts.size),
            'observed_count': int(obs_count),
            'forecast_count_mean': float(counts.mean()),
            'forecast_count_std': float(counts.std(ddof=0)),
            'forecast_count_rmse': rmse,
            'forecast_count_q025': float(q025),
            'forecast_count_q975': float(q975),
            'obs_within_95pi': coverage,
        }
    )
    return result


def _write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')


def _write_csv(path: Path, rows: Iterable[dict[str, Any]]) -> None:
    rows = list(rows)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text('', encoding='utf-8')
        return
    keys = sorted({k for row in rows for k in row.keys()})
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)

In [8]:
set_seed(SEED)

checkpoint_path = CHECKPOINT_PATH.expanduser().resolve()
if not checkpoint_path.exists():
    raise FileNotFoundError(f'checkpoint file not found: {checkpoint_path}')

model, ckpt_args = load_and_prepare_model(
    checkpoint_path=checkpoint_path,
    device=DEVICE,
    compile=COMPILE_MODEL,
)

base_cfg = dict(getattr(ckpt_args, 'catalog_cfg', {}) or {})
dataset_cfg_map = _load_dataset_cfg_map(DATASET_CFG_FILE if DATASET_CFG_FILE is not None else None)

global_cfg = dict(GLOBAL_CATALOG_CFG)

out_json = OUTPUT_JSON if OUTPUT_JSON is not None else checkpoint_path.parent / 'batch_eval_metrics.json'
out_csv = OUTPUT_CSV if OUTPUT_CSV is not None else checkpoint_path.parent / 'batch_eval_metrics.csv'
out_json = Path(out_json).expanduser().resolve()
out_csv = Path(out_csv).expanduser().resolve()

results: list[dict[str, Any]] = []
for dataset_name in DATASETS:
    cfg = dict(base_cfg)
    cfg.update(global_cfg)
    cfg.update(dataset_cfg_map.get(dataset_name, {}))
    try:
        row = evaluate_one_dataset(
            model=model,
            device=DEVICE,
            dataset_name=dataset_name,
            catalog_cfg=cfg,
            t_forecast=T_FORECAST,
            forecast_duration=FORECAST_DURATION,
            num_samples=NUM_SAMPLES,
            samples_per_batch=SAMPLES_PER_BATCH,
            sample_max_length=SAMPLE_MAX_LENGTH,
            seed=SEED,
            skip_forecast=SKIP_FORECAST,
        )
        row['status'] = 'ok'
        results.append(row)
        print(
            f"[ok] dataset={dataset_name} "
            f"test_nll={row.get('test_nll')} "
            f"obs={row.get('observed_count')} "
            f"fc_mean={row.get('forecast_count_mean')}"
        )
    except Exception as exc:
        err = {
            'dataset': dataset_name,
            'status': 'error',
            'error_type': type(exc).__name__,
            'error_message': str(exc),
        }
        results.append(err)
        print(f"[error] dataset={dataset_name}: {type(exc).__name__}: {exc}")

payload = {
    'checkpoint_path': str(checkpoint_path),
    'device': str(DEVICE),
    'compile': bool(COMPILE_MODEL),
    'seed': int(SEED),
    'datasets': DATASETS,
    'results': results,
}

_write_json(out_json, payload)
_write_csv(out_csv, results)

print(f'[saved] json={out_json}')
print(f'[saved] csv={out_csv}')

results_df = pd.DataFrame(results)
results_df

[ok] dataset=SSFS test_nll=-722.0640258789062 obs=None fc_mean=None


/root/autodl-tmp/em_eqf/src/data/sequence.py:246: UserWarning: Found 1 zero inter-event times in the sequence. This violates fundamental assumptions of TPP models and may lead to incorrect log-likelihood values.
  warnings.warn(


[ok] dataset=Basel test_nll=0.3662920296192169 obs=None fc_mean=None
[ok] dataset=PNR_1z test_nll=-1141.90380859375 obs=None fc_mean=None
[saved] json=/root/autodl-tmp/em_eqf/checkpoints/etas_20260312-154442/batch_eval_metrics.json
[saved] csv=/root/autodl-tmp/em_eqf/checkpoints/etas_20260312-154442/batch_eval_metrics.csv


,dataset,catalog_registry_name,catalog_init_kwargs,test_nll,num_events_test,test_t_start,test_t_end,test_t_nll_start,status
0,SSFS,SSFS-Standard,{'root_dir': '/root/autodl-tmp/em_eqf/data/SSF...,-722.064026,2392,0.0,16.896528,0.000000,ok
1,Basel,Basel-Standard,{'root_dir': '/root/autodl-tmp/em_eqf/data/Bas...,0.366292,1211,0.0,4267.265625,3627.174896,ok
2,PNR_1z,PNR_1z-Standard,{'root_dir': '/root/autodl-tmp/em_eqf/data/PNR...,-1141.903809,6140,0.0,64.208328,59.666667,ok


## Notes
- If forecast samples explode for some datasets, reduce `FORECAST_DURATION` or set `SAMPLE_MAX_LENGTH`.
- Use `SKIP_FORECAST=True` for fast NLL-only checks before full sampling.